## v2.2 — v1.2 + feature engineering (Logistic Regression, Linear SVM, Random Forest)

Same NaN-handling setup as [v1.2.ipynb](./v1.2.ipynb) (`BankChurnImputer`:
`country`→`"Unknown"`, `acc_balance`→median-by-country, `credit_score`→median,
`prod_count`→`RandomForestClassifier`, all fit only on training-fold data).
This notebook adds the feature engineering from
`docs/Nishkarsh/feature_engineering.md` on top, to compare against the v1.2
baseline:

```text
LogisticRegression: OOF F1=0.6325
LinearSVC          : OOF F1=0.6323
RandomForest        : OOF F1=0.6458
```


In [1]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42


In [2]:
train = pd.read_csv('../../data/train.csv')
test = pd.read_csv('../../data/test.csv')
print("train:", train.shape, "test:", test.shape)
print(train.isna().sum())


train: (90000, 14) test: (30000, 13)
id                     0
customer_id            0
last_name              0
credit_score        9556
country             6021
gender                 0
age                    0
tenure                 0
acc_balance         7257
prod_count          4863
has_card               0
is_active              0
estimated_salary       0
exit_status            0
dtype: int64


In [3]:
ID_COLS = ['id', 'customer_id', 'last_name']
TARGET = 'exit_status'
PROD_COUNT_IMPUTE_FEATURES = [
    'age', 'is_active', 'acc_balance', 'country', 'credit_score',
    'has_card', 'estimated_salary', 'tenure',
]


def make_X(df):
    return df.drop(columns=[c for c in ID_COLS + [TARGET] if c in df.columns])


### Feature engineering (from `docs/Nishkarsh/feature_engineering.md`)

New columns, added on top of the imputed data, never replacing the originals:

* `balance_zero` — `acc_balance == 0` as an explicit flag. The column is
  bimodal with a huge spike at exactly zero; a flag makes that discontinuity
  explicit instead of relying on the model to discover it (NaN-aware: rows
  with an unknown balance get an unknown flag, not a fabricated 0/1).
* `country_gender` — interaction of `country` and `gender`, since churn by
  country is itself gender-skewed in this dataset.
* `active_prodcount` — interaction of `is_active` and `prod_count`, both
  independently strong churn predictors.
* `balance_salary_ratio`, `balance_age_ratio` — `acc_balance / estimated_salary`
  and `acc_balance / age`. Minor but consistent contributors in public
  notebooks on the source dataset. Division guards against a zero
  denominator (doesn't occur in this data, but cheap to guard).
* `credit_score_band` — standard FICO-style bands (Poor/Fair/Good/VeryGood/
  Excellent). Included for completeness, but `credit_score` had ~0
  correlation with everything in the original EDA, so this one is a
  low-priority, low-expectation addition.

`prod_count` itself is already treated as categorical (one-hot / native
category dtype) rather than numeric in v1.1/v1.2, since its relationship with
churn is strongly non-monotonic (2 is the *safest* count) — that's carried
over unchanged, it isn't new here.

Recall the biggest documented finding on this dataset: a 300+-pipeline sweep
on the source Kaggle competition found engineered features gave only small
gains, with model/encoding choice mattering more. So the honest goal here is
to measure the actual delta against the v1 baseline, not to assume this helps.


In [4]:
def add_engineered_features(df):
    df = df.copy()

    df['balance_zero'] = np.where(
        df['acc_balance'].isna(), np.nan, (df['acc_balance'] == 0).astype(float)
    )
    df['country_gender'] = np.where(
        df['country'].isna(), np.nan,
        df['country'].astype(str) + '_' + df['gender'].astype(str)
    )
    df['active_prodcount'] = np.where(
        df['prod_count'].isna(), np.nan,
        df['is_active'].astype(int).astype(str) + '_' + df['prod_count'].astype('Int64').astype(str)
    )
    df['balance_salary_ratio'] = df['acc_balance'] / df['estimated_salary'].replace(0, np.nan)
    df['balance_age_ratio'] = df['acc_balance'] / df['age'].replace(0, np.nan)

    credit_bins = [-np.inf, 579, 669, 739, 799, np.inf]
    credit_labels = ['Poor', 'Fair', 'Good', 'VeryGood', 'Excellent']
    df['credit_score_band'] = pd.cut(df['credit_score'], bins=credit_bins, labels=credit_labels)

    return df


In [5]:
NUMERIC_COLS = [
    'credit_score', 'age', 'tenure', 'acc_balance', 'has_card', 'is_active', 'estimated_salary',
    'balance_zero', 'balance_salary_ratio', 'balance_age_ratio',
]
CATEGORICAL_COLS = ['country', 'gender', 'prod_count', 'country_gender', 'active_prodcount', 'credit_score_band']


### `BankChurnImputer`

Unchanged from v1.2 -- fills `country`/`acc_balance`/`credit_score`/`prod_count`
on the *original* columns, before feature engineering runs. Since every
source column is guaranteed filled by the time `add_engineered_features` runs
here, none of the new columns can pick up a stray `NaN` (`OneHotEncoder`
below also doesn't need a fixed category list the way v2.1's boosting models
do -- `handle_unknown='ignore'` handles any category seen in a fold that
wasn't seen elsewhere).


In [6]:
class BankChurnImputer(BaseEstimator, TransformerMixin):
    def __init__(self, prod_count_features=PROD_COUNT_IMPUTE_FEATURES, random_state=RANDOM_STATE):
        self.prod_count_features = prod_count_features
        self.random_state = random_state

    def fit(self, X, y=None):
        X = X.copy()
        X['country'] = X['country'].fillna('Unknown')
        self.balance_median_by_country_ = X.groupby('country')['acc_balance'].median()
        self.balance_global_median_ = X['acc_balance'].median()
        self.credit_score_median_ = X['credit_score'].median()

        X['acc_balance'] = X['acc_balance'].fillna(X['country'].map(self.balance_median_by_country_))
        X['acc_balance'] = X['acc_balance'].fillna(self.balance_global_median_)
        X['credit_score'] = X['credit_score'].fillna(self.credit_score_median_)

        known = X.dropna(subset=['prod_count'])
        Xk = pd.get_dummies(known[self.prod_count_features], columns=['country'])
        self.prod_count_columns_ = Xk.columns
        yk = known['prod_count'].astype(int)

        self.prod_count_model_ = RandomForestClassifier(
            n_estimators=300, max_depth=None, min_samples_leaf=5,
            random_state=self.random_state, n_jobs=-1,
        )
        self.prod_count_model_.fit(Xk, yk)
        return self

    def transform(self, X):
        X = X.copy()
        X['country'] = X['country'].fillna('Unknown')
        X['acc_balance'] = X['acc_balance'].fillna(X['country'].map(self.balance_median_by_country_))
        X['acc_balance'] = X['acc_balance'].fillna(self.balance_global_median_)
        X['credit_score'] = X['credit_score'].fillna(self.credit_score_median_)

        missing = X['prod_count'].isna()
        if missing.any():
            Xm = pd.get_dummies(X.loc[missing, self.prod_count_features], columns=['country'])
            Xm = Xm.reindex(columns=self.prod_count_columns_, fill_value=0)
            X.loc[missing, 'prod_count'] = self.prod_count_model_.predict(Xm)
        return X


### OOF cross-validation + F1 threshold search

Same harness as v1.2, with `add_engineered_features` inserted between the
mandatory imputer and the shared `ColumnTransformer` (scaling + one-hot).


In [7]:
def make_preprocessor():
    return ColumnTransformer([
        ('num', StandardScaler(), NUMERIC_COLS),
        ('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL_COLS),
    ])


def oof_threshold_search(build_model_fn, X, y, score_fn='predict_proba', n_splits=5, random_state=RANDOM_STATE):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    oof = np.zeros(len(X))

    for tr_idx, va_idx in skf.split(X, y):
        X_tr, X_va = X.iloc[tr_idx].copy(), X.iloc[va_idx].copy()
        y_tr = y.iloc[tr_idx]

        imp = BankChurnImputer()
        imp.fit(X_tr)
        X_tr, X_va = imp.transform(X_tr), imp.transform(X_va)
        X_tr, X_va = add_engineered_features(X_tr), add_engineered_features(X_va)

        pre = make_preprocessor()
        X_tr_enc = pre.fit_transform(X_tr)
        X_va_enc = pre.transform(X_va)

        model = build_model_fn()
        sw = compute_sample_weight('balanced', y_tr)
        model.fit(X_tr_enc, y_tr, sample_weight=sw)

        if score_fn == 'predict_proba':
            oof[va_idx] = model.predict_proba(X_va_enc)[:, 1]
        else:
            oof[va_idx] = model.decision_function(X_va_enc)

    if score_fn == 'predict_proba':
        thresholds = np.linspace(0.02, 0.98, 97)
    else:
        thresholds = np.linspace(oof.min(), oof.max(), 97)
    f1s = [f1_score(y, oof > t) for t in thresholds]
    best = int(np.argmax(f1s))
    return oof, thresholds[best], f1s[best]


In [8]:
X = make_X(train)
y = train[TARGET]

def build_logreg():
    return LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)

_, thr_lr, f1_lr = oof_threshold_search(build_logreg, X, y, score_fn='predict_proba')
print(f"LogisticRegression: OOF F1={f1_lr:.4f}  best_threshold={thr_lr:.2f}")


LogisticRegression: OOF F1=0.6340  best_threshold=0.63


In [9]:
def build_svm():
    return LinearSVC(max_iter=5000, random_state=RANDOM_STATE)

_, thr_svm, f1_svm = oof_threshold_search(build_svm, X, y, score_fn='decision_function')
print(f"LinearSVC: OOF F1={f1_svm:.4f}  best_threshold={thr_svm:.2f}")


LinearSVC: OOF F1=0.6311  best_threshold=0.24


In [10]:
def build_rf():
    return RandomForestClassifier(
        n_estimators=400, max_depth=None, min_samples_leaf=3,
        random_state=RANDOM_STATE, n_jobs=-1,
    )

_, thr_rf, f1_rf = oof_threshold_search(build_rf, X, y, score_fn='predict_proba')
print(f"RandomForest: OOF F1={f1_rf:.4f}  best_threshold={thr_rf:.2f}")


RandomForest: OOF F1=0.6440  best_threshold=0.49


In [11]:
results = pd.DataFrame([
    {'model': 'LogisticRegression', 'oof_f1': f1_lr, 'threshold': thr_lr, 'score_fn': 'predict_proba'},
    {'model': 'LinearSVC', 'oof_f1': f1_svm, 'threshold': thr_svm, 'score_fn': 'decision_function'},
    {'model': 'RandomForest', 'oof_f1': f1_rf, 'threshold': thr_rf, 'score_fn': 'predict_proba'},
]).sort_values('oof_f1', ascending=False).reset_index(drop=True)

v1_2_baseline = pd.DataFrame([
    {'model': 'LogisticRegression', 'v1_2_oof_f1': 0.6325},
    {'model': 'LinearSVC', 'v1_2_oof_f1': 0.6323},
    {'model': 'RandomForest', 'v1_2_oof_f1': 0.6458},
])
comparison = results.merge(v1_2_baseline, on='model')
comparison['delta_vs_v1_2'] = comparison['oof_f1'] - comparison['v1_2_oof_f1']
comparison


,model,oof_f1,threshold,score_fn,v1_2_oof_f1,delta_vs_v1_2
0,RandomForest,0.643953,0.490000,predict_proba,0.6458,-0.001847
1,LogisticRegression,0.633973,0.630000,predict_proba,0.6325,0.001473
2,LinearSVC,0.631149,0.235271,decision_function,0.6323,-0.001151


### Final fit + submission

Refits whichever model scored best above on the full training set and writes
a submission.


In [12]:
import os

BUILDERS = {'LogisticRegression': build_logreg, 'LinearSVC': build_svm, 'RandomForest': build_rf}
best_row = results.iloc[0]
best_name = best_row['model']
best_threshold = best_row['threshold']
best_score_fn = best_row['score_fn']
print(f"Refitting best model: {best_name} (OOF F1={best_row['oof_f1']:.4f})")

X_train_final = make_X(train)
y_train_final = train[TARGET]
X_test_final = make_X(test)

final_imputer = BankChurnImputer()
final_imputer.fit(X_train_final)
X_train_final = add_engineered_features(final_imputer.transform(X_train_final))
X_test_final = add_engineered_features(final_imputer.transform(X_test_final))

final_pre = make_preprocessor()
X_train_enc = final_pre.fit_transform(X_train_final)
X_test_enc = final_pre.transform(X_test_final)

final_model = BUILDERS[best_name]()
sw_final = compute_sample_weight('balanced', y_train_final)
final_model.fit(X_train_enc, y_train_final, sample_weight=sw_final)

if best_score_fn == 'predict_proba':
    test_scores = final_model.predict_proba(X_test_enc)[:, 1]
else:
    test_scores = final_model.decision_function(X_test_enc)
test_pred = (test_scores > best_threshold).astype(int)

os.makedirs('outputs', exist_ok=True)
submission = pd.DataFrame({'id': test['id'], 'exit_status': test_pred})
submission.to_csv(f'outputs/v2_2_{best_name.lower()}_submission.csv', index=False)
submission.head()


Refitting best model: RandomForest (OOF F1=0.6440)


,id,exit_status
0,0,0
1,1,1
2,2,1
3,3,0
4,4,0
